# 02 · Classical Supervised Learning

Compare classification and regression model families using a shared split, preprocessing pipeline, cross-validation, and imbalance diagnostics.

**How to use this notebook**

1. Run cells from top to bottom.
2. Start with the CI profile; switch to `portfolio` only after the workflow is clear.
3. Change one parameter at a time and record the effect rather than chasing a high score.

> All records are deterministic synthetic data. Results are educational evidence, not a
> production or financial decision system.

In [ ]:
# Shared local/Colab bootstrap. Run this cell first.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "codex/colab-notebook-first",
                "--single-branch",
                "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
    %pip install -q -e ".[full,dev]"
else:
    here = Path.cwd().resolve()
    if here.name == "notebooks":
        os.chdir(here.parent)

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").exists(), "Please open the repository root or notebooks folder."
print(f"environment={'Colab' if IN_COLAB else 'local/VS Code'}")
print(f"repository={ROOT}")

In [ ]:
from pathlib import Path

import pandas as pd

from rookie_lab.classical.experiment import run

PROFILE = "ci"
metrics = run(Path(f"data/generated/{PROFILE}"), PROFILE)
classification = pd.DataFrame(metrics["classification"]).T.sort_values("roc_auc", ascending=False)
regression = pd.DataFrame(metrics["regression"]).T.sort_values("r2", ascending=False)
display(classification)
display(regression)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
classification["roc_auc"].sort_values().plot.barh(ax=axes[0], title="Churn ROC-AUC")
regression["r2"].sort_values().plot.barh(ax=axes[1], title="Spend R²")
plt.tight_layout()
print("Cross-validation", metrics["cross_validation"])
print("Imbalance", metrics["class_imbalance"])
print("XGBoost backend", metrics["xgboost_backend"])

## Inspect the reusable preprocessing graph

In [ ]:
from rookie_lab.classical.preprocessing import preprocessor

prep = preprocessor()
print(prep)
print("Numeric: median imputation + scaling")
print("Categorical: most-frequent imputation + one-hot encoding")

## Interpretation challenge

Why can accuracy be near 0.97 while F1 is near zero? Compare ROC-AUC and PR-AUC before selecting a model.